# Capa 5 — Machine Learning: Riesgo de Abandono Temprano
## Proyecto: Customer Analytics — "Super Compras"

**Metodología:** 6 Capas (Estrategia → Preparación → EDA → Análisis Clásico → **ML** → Entregable)

**Nivel de la pirámide:** Nivel III (Predictivo), acotado — definido en la Capa 1 para responder
directamente a los objetivos 3 y 4 del negocio: *"entender la retención"* y *"activar clientes
en riesgo de pérdida"*.

**Pregunta que responde este notebook:** con solo el comportamiento del primer mes de un
cliente (M1) y sus datos sociodemográficos, ¿se puede anticipar si va a seguir comprando más
adelante?


## 0. Variables: qué entra al modelo y qué NO (y por qué)

Este es el paso más importante del notebook, y se decide ANTES de tocar el algoritmo.

**Variables permitidas (conocidas en el momento de predecir, justo al terminar M1):**
- Sociodemográficas: `Genero`, `Nivel_Educativo`, `Ocupacion`, `Tamano_Hogar`, `Edad`, `CAC`
- Comportamiento de M1: `gasto_M1` (€ gastados en el primer mes), `frecuencia_M1` (nº de
  compras en el primer mes), `ticket_medio_M1` (gasto_M1 / frecuencia_M1)

**Variables explícitamente EXCLUIDAS (circularidad — decidido en la Capa 1):**
`Monetary`, `Recency`, `Frequency`, `r`, `f`, `m`, `RFM`, `Segmento`, `ltv`. Todas estas se
calculan usando el histórico COMPLETO del cliente (año completo o toda la ventana de 6 meses),
es decir, usan información posterior al momento en que se necesita hacer la predicción (justo
después de M1). Usarlas sería tan artificial como "predecir" el resultado de un partido
enseñándole al modelo el marcador final.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              classification_report, accuracy_score)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42
RUTA = "./"   # carpeta con tablon_analitico_reconstruido.csv y cliente_nivel_eda.csv


## 1. Construcción del dataset de modelado

Se parte de `tablon_analitico_reconstruido.csv` (transacciones dentro de la ventana de 6 meses)
para calcular el comportamiento de M1 de cada cliente, y se define la variable objetivo: **¿el
cliente tiene alguna transacción en M3?** (`activo_M3`, 1/0). Como robustez, se calcula también
`activo_M6` (horizonte más lejano, más difícil de predecir).

In [ ]:
t = pd.read_csv(RUTA + "tablon_analitico_reconstruido.csv", parse_dates=["Fecha", "Fecha_Nacimiento"])
cli = pd.read_csv(RUTA + "cliente_nivel_eda.csv")

m1 = (t[t.MesCliente == "M1"]
      .groupby("ID_Cliente")
      .agg(gasto_M1=("Total_Venta", "sum"), frecuencia_M1=("ID_Transaccion", "nunique"))
      .reset_index())

activos_M3 = set(t[t.MesCliente == "M3"]["ID_Cliente"].unique())
activos_M6 = set(t[t.MesCliente == "M6"]["ID_Cliente"].unique())

df = cli.merge(m1, on="ID_Cliente", how="left")
df["gasto_M1"] = df["gasto_M1"].fillna(0)
df["frecuencia_M1"] = df["frecuencia_M1"].fillna(0)
df["ticket_medio_M1"] = (df["gasto_M1"] / df["frecuencia_M1"].replace(0, np.nan)).fillna(0)
df["activo_M3"] = df["ID_Cliente"].isin(activos_M3).astype(int)
df["activo_M6"] = df["ID_Cliente"].isin(activos_M6).astype(int)

print("Dataset de modelado:", df.shape[0], "clientes")
df[["ID_Cliente", "gasto_M1", "frecuencia_M1", "ticket_medio_M1", "activo_M3", "activo_M6"]].head()


## 2. Balance de clases — el hallazgo más importante antes de modelar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, tgt in zip(axes, ["activo_M3", "activo_M6"]):
    df[tgt].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#C62828", "#2E7D32"])
    ax.set_xticklabels(["No vuelve (0)", "Vuelve (1)"], rotation=0)
    ax.set_title(f"{tgt}  ({df[tgt].mean():.1%} vuelve)")
plt.tight_layout()
plt.show()


**Solo el 16,6% de los clientes tiene alguna compra en M3, y solo el 9,9% en M6.** Es decir:
de cada 10 clientes nuevos, más de 8 no vuelven a comprar en los 3 meses siguientes a su primera
compra. Esto ya es, en sí mismo, uno de los hallazgos de negocio más accionables de todo el
proyecto — independientemente de lo que el modelo consiga predecir.

Con esta proporción (83,4% / 16,6%), el problema está **desbalanceado**: un modelo perezoso que
prediga siempre "no vuelve" acertaría el 83,4% de las veces sin aprender nada útil. Por eso se
usa `class_weight="balanced"` en ambos modelos (penaliza más los errores sobre la clase
minoritaria) en vez de dejar que el modelo se limite a predecir la clase mayoritaria. No se usa
sobremuestreo tipo SMOTE porque no añade información real — con `class_weight` se consigue el
mismo objetivo sin inventar datos sintéticos.

## 3. Protocolo de validación: partición 70/30 estratificada

Estratificada = se mantiene la misma proporción de "vuelve"/"no vuelve" en entrenamiento y test,
para que la evaluación sea representativa.

In [ ]:
cat_features = ["Genero", "Nivel_Educativo", "Ocupacion", "Tamano_Hogar"]
num_features = ["CAC", "gasto_M1", "frecuencia_M1", "ticket_medio_M1", "Edad"]

X = pd.get_dummies(df[cat_features + num_features], columns=cat_features, drop_first=True)
y = df["activo_M3"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
print("Entrenamiento:", Xtr.shape, " Test:", Xte.shape)
print("Balance train:", ytr.mean().round(3), " Balance test:", yte.mean().round(3))


## 4. Modelos: Regresión Logística (interpretable) vs. Random Forest (flexible)

Se entrenan los dos, como pide la metodología del proyecto para clasificación: un modelo lineal
de referencia y un modelo de ensamblado más potente.

In [ ]:
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xte_s = scaler.transform(Xte)

logreg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(Xtr_s, ytr)
proba_lr = logreg.predict_proba(Xte_s)[:, 1]
auc_lr = roc_auc_score(yte, proba_lr)

rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced",
                             random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xtr, ytr)
proba_rf = rf.predict_proba(Xte)[:, 1]
auc_rf = roc_auc_score(yte, proba_rf)

print(f"AUC-ROC Regresión Logística: {auc_lr:.4f}")
print(f"AUC-ROC Random Forest:       {auc_rf:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for proba, auc, nombre, color in [(proba_lr, auc_lr, "Regresión Logística", "#4A90D9"),
                                    (proba_rf, auc_rf, "Random Forest", "#2E7D32")]:
    fpr, tpr, _ = roc_curve(yte, proba)
    ax.plot(fpr, tpr, label=f"{nombre} (AUC={auc:.3f})", color=color, lw=2)
ax.plot([0, 1], [0, 1], color="grey", ls="--", lw=1, label="Azar (AUC=0.5)")
ax.set_xlabel("Falsos positivos")
ax.set_ylabel("Verdaderos positivos")
ax.set_title("Curva ROC — activo_M3")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Resultado frente al criterio de éxito de la Capa 1

**La Capa 1 fijó como métrica de éxito: AUC-ROC > 0,65.** Con los datos disponibles, **ningún
modelo alcanza ese umbral** (Random Forest, el mejor de los dos, llega a ~0,61). Se reporta así,
sin maquillar el resultado — el propio proyecto identificó en la auditoría de la aplicación
anterior (ver bitácora) el problema de inflar artificialmente un AUC, y aquí se hace justo lo
contrario: mostrar el número real y explicar honestamente por qué es el que es.

**Por qué es un resultado razonable, no un error del proceso:** se está prediciendo con una sola
variable de comportamiento (el primer mes) sobre una base de clientes donde el 83% simplemente
no vuelve — con tan poca información y un fenómeno tan dominado por el "no", un AUC de 0,61 sí
representa señal real (mejor que el azar, 0,50), aunque modesta. Se retoma en la Sección 7 lo
que haría falta para mejorarlo.

## 6. Modelo final: Random Forest — métricas completas

In [ ]:
pred_rf = rf.predict(Xte)
print(classification_report(yte, pred_rf, digits=3, target_names=["No vuelve (0)", "Vuelve (1)"]))

cm = confusion_matrix(yte, pred_rf)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=ax,
            xticklabels=["Pred: No vuelve", "Pred: Vuelve"],
            yticklabels=["Real: No vuelve", "Real: Vuelve"])
ax.set_title("Matriz de confusión — Random Forest")
plt.tight_layout()
plt.show()

acc_modelo = accuracy_score(yte, pred_rf)
acc_naive = accuracy_score(yte, np.zeros_like(yte))
print(f"Accuracy del modelo: {acc_modelo:.3f}")
print(f"Accuracy de predecir siempre \"no vuelve\" (naive): {acc_naive:.3f}")


**Punto importante para no leer mal esta tabla: el modelo tiene MENOS accuracy (73,5%) que
el naive de predecir siempre "no vuelve" (83,4%).** Esto NO significa que el modelo sea peor —
significa que accuracy es una métrica engañosa cuando las clases están tan desbalanceadas: el
naive "acierta mucho" simplemente por apostar siempre a la opción mayoritaria, pero no identifica
ni a un solo cliente que sí vuelve (recall de la clase 1 = 0%). El modelo, con `class_weight`,
sacrifica accuracy general a cambio de detectar un 34,4% de los clientes que sí vuelven (recall
de la clase 1) — que es justo lo que el negocio necesita: encontrar a esos clientes, no ignorarlos.

## 7. ¿Qué variables pesan más? (Random Forest vs. Regresión Logística)

In [ ]:
imp_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
coef_lr = pd.Series(logreg.coef_[0], index=X.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
imp_rf.plot(kind="barh", ax=axes[0], color="#2E7D32")
axes[0].set_title("Importancia de variables — Random Forest")

colors = ["#C62828" if v < 0 else "#2E7D32" for v in coef_lr.values]
coef_lr.plot(kind="barh", ax=axes[1], color=colors)
axes[1].set_title("Coeficientes — Regresión Logística\n(rojo = asociado a NO volver)")
plt.tight_layout()
plt.show()


**Lectura de negocio:**
- **`frecuencia_M1` es, con diferencia, la variable más importante en ambos modelos** —
  cuántas veces compró el cliente en su primer mes importa mucho más que cuánto se gastó.
  Un cliente que vuelve 3 veces en su primer mes (aunque compre poco cada vez) es mejor señal
  de que va a seguir comprando que uno que gasta mucho en una sola visita.
- `CAC` y `gasto_M1` aportan también señal relevante en el Random Forest.
- **Las variables sociodemográficas (Género, Nivel_Educativo, Ocupacion, Tamano_Hogar) apenas
  pesan** — coherente con el hallazgo de la Capa 4 (Cramér's V negligible en todos los cruces
  con "En Riesgo"). El comportamiento temprano predice mejor que el perfil.

## 8. Valor práctico pese al AUC modesto: análisis por deciles

Aunque el AUC no llega al umbral fijado, el modelo puede seguir siendo útil si **ordena bien**
a los clientes de más a menos probable que vuelvan — que es, en la práctica, lo que necesita
una campaña de marketing con presupuesto limitado (a quién llamar primero).

In [ ]:
resultado = pd.DataFrame({"y_real": yte.values, "score_modelo": proba_rf})
resultado["decil"] = pd.qcut(resultado["score_modelo"], 10, labels=False, duplicates="drop")

tasa_global = resultado["y_real"].mean()
tabla_deciles = (resultado.groupby("decil")
                 .agg(clientes=("y_real", "size"), tasa_retorno_real=("y_real", "mean"))
                 .sort_index(ascending=False))
tabla_deciles.index = [f"Decil {10-i} ({'más' if i==0 else '' }probable)" if i == 0 else f"Decil {10-i}"
                       for i in range(len(tabla_deciles))]
tabla_deciles["lift_vs_azar"] = (tabla_deciles["tasa_retorno_real"] / tasa_global).round(2)
tabla_deciles["tasa_retorno_real"] = (tabla_deciles["tasa_retorno_real"] * 100).round(1)
tabla_deciles


**El 10% de clientes que el modelo puntúa como más propensos a volver tiene una tasa de
retorno real del 33,9%, frente al 16,6% de media — un lift de 2,04x.** Traducido a negocio: si
el equipo de marketing usa este modelo para priorizar a quién dirigir una campaña de activación
justo después de M1, doblaría la eficacia de acertar con quién sí va a volver, comparado con
elegir al azar. Esto es exactamente el tipo de uso práctico que el AUC, por sí solo, no deja ver.

## 9. Robustez: ¿el mismo patrón se repite prediciendo M6?

Se repite el mismo pipeline (mismos features, mismo protocolo) cambiando el objetivo a
`activo_M6`, un horizonte más lejano y más difícil.

In [ ]:
resumen_final = []
for target in ["activo_M3", "activo_M6"]:
    y_t = df[target]
    Xtr_t, Xte_t, ytr_t, yte_t = train_test_split(X, y_t, test_size=0.30, random_state=RANDOM_STATE, stratify=y_t)

    scaler_t = StandardScaler()
    Xtr_ts, Xte_ts = scaler_t.fit_transform(Xtr_t), scaler_t.transform(Xte_t)
    lr_t = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE).fit(Xtr_ts, ytr_t)
    auc_lr_t = roc_auc_score(yte_t, lr_t.predict_proba(Xte_ts)[:, 1])

    rf_t = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced",
                                    random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr_t, ytr_t)
    auc_rf_t = roc_auc_score(yte_t, rf_t.predict_proba(Xte_t)[:, 1])

    resumen_final.append([target, f"{y_t.mean():.1%}", round(auc_lr_t, 4), round(auc_rf_t, 4)])

resumen_final_df = pd.DataFrame(resumen_final, columns=["Objetivo", "% que vuelve", "AUC LogReg", "AUC Random Forest"])
resumen_final_df


**El patrón se confirma y se acentúa: predecir M6 es más difícil que predecir M3** (menos
clientes positivos, 9,9% vs 16,6%, y AUC ligeramente menor). Es coherente: cuanto más lejos en
el tiempo se intenta predecir con una sola señal de comportamiento inicial, menos información
queda. Esto no es un fallo del modelo — es una limitación real de trabajar solo con M1.

## 10. Conclusiones de la Capa 5 y qué haría falta para mejorar

**Resultado honesto:** el modelo no alcanza el AUC>0,65 fijado como éxito en la Capa 1. Alcanza
~0,61 (M3) y ~0,59-0,60 (M6) — mejor que el azar, con una variable (`frecuencia_M1`) claramente
dominante, pero insuficiente como sistema de decisión automática.

**Lo que sí es utilizable ya:** el ranking por decil (Sección 8) — un lift de ~2x en el decil
superior es un resultado real y accionable para priorizar campañas de activación temprana, aunque
no sirva para decidir de forma binaria y automática a quién dar de baja del presupuesto de
marketing.

**Qué haría falta para superar el umbral (recomendación para una siguiente iteración, fuera del
alcance actual):**
- Señal de producto/categoría comprada en M1 (no solo importe y frecuencia) — qué compra, no
  solo cuánto.
- Variables de canal/campaña de captación (cómo llegó el cliente), no disponibles en este dataset.
- Un histórico más largo de "primeras semanas" (no solo mes 1 agregado) podría capturar patrones
  de recurrencia más finos (p. ej., ¿volvió en los primeros 7 días?).

**Siguiente paso:** `Capa 6` — informe de conclusiones y recomendaciones, integrando los
hallazgos de las Capas 3, 4 y 5 en un documento único para la dirección de "Super Compras".

In [ ]:
df.to_csv("dataset_capa5_final.csv", index=False)
resumen_final_df.to_csv("resumen_modelos_capa5.csv", index=False)
print("Guardado: dataset_capa5_final.csv, resumen_modelos_capa5.csv")
